# 第 9 课：一维卷积、局部上下文与时间下采样

上一课的逐帧 MLP 每次只看一帧。本课使用 Conv1d 在时间轴上滑动，让输出同时参考前后帧，并用 stride 减少时间长度。

路线：滑动窗口直觉 → kernel/stride/padding → Conv1d shape → 局部上下文 → 两层下采样 → 更新 length 与 mask。

<!-- course-upgrade-v2 -->
## 学习导航

| 项目 | 内容 |
|---|---|
| 所属阶段 | 张量与编码器 |
| 建议投入 | 2～4 小时，可分 2～3 次完成 |
| 前置要求 | 完成第 8 课；如果前测低于 2/3，先回看上一课小结 |
| 本课核心 | Conv1d、stride 下采样、感受野 |
| 完成标准 | 能口头解释核心概念；独立完成强化题；从空白重写核心函数 |

高效顺序：**先回答前测 → 预测代码结果 → 再运行 → 修改一个变量 → 关闭答案复现 → 次日回忆。**


<!-- course-upgrade-v2 -->
## 课前诊断（先不要运行代码）

1. 分别用一句话解释：Conv1d、stride 下采样、感受野。
2. 画出这三个概念之间的输入—输出关系。
3. 写下你最不确定的一点，并给出一个暂时猜测。

自评：答对 0～1 题先复习前置课；答对 2 题可以正常学习；3 题都能讲清楚则直接挑战代码和迁移题。


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
import librosa
import torch
from torch import nn
import torch.nn.functional as F
import ipywidgets as widgets

ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
PARTS=ROOT/'data'/'spoken_digits_parts'
torch.manual_seed(9)

## 1. 为什么需要时间上下文？

一个 25 ms Log-Mel 帧只描述很短的声音。音素和音节由连续变化形成，判断当前帧时通常需要附近帧。

逐帧 Linear：每个输出只看同一时刻。

kernel size=3 的 Conv1d：每个输出可综合左、中、右三个时间位置。

## 2. 最简单的一维滑动计算

输入序列 `[1,2,3,4,5,6,7,8]`，kernel `[1,0,-1]`。kernel 每次覆盖三个数字并计算加权和。PyTorch Conv1d 实际使用交叉相关，即 kernel 不翻转。

In [ ]:
sequence=torch.arange(1,9,dtype=torch.float32).reshape(1,1,-1)
kernel=torch.tensor([1.,0.,-1.]).reshape(1,1,-1)
output=F.conv1d(sequence,kernel,stride=1,padding=0)
print('input:',sequence.flatten().tolist())
print('kernel:',kernel.flatten().tolist())
print('output:',output.flatten().tolist())
print('第一个输出 = 1×1 + 2×0 + 3×(-1) =',float(output[0,0,0]))

In [ ]:
fig,axes=plt.subplots(2,1,figsize=(10,5))
axes[0].stem(np.arange(8),sequence.flatten().numpy());axes[0].set_title('Input sequence')
axes[1].stem(np.arange(output.shape[-1]),output.flatten().numpy());axes[1].set_title('Conv1d output')
for ax in axes:ax.set_xlabel('Time index');ax.grid(alpha=0.2)
plt.tight_layout();plt.show()

### 练习 1

1. 第二个滑动窗口包含哪三个数字？
2. 第二个输出是多少？
3. 输入长度 8、kernel 3、不 padding 时，为什么输出长度只有 6？

## 3. 交互观察 kernel、stride 和 padding

- kernel size：一次覆盖多少位置。
- stride：每次移动多少位置。
- padding：在输入两侧补多少位置。

In [ ]:
def draw_windows(length=12,kernel_size=3,stride=1,padding=0):
    padded_length=length+2*padding
    starts=list(range(0,padded_length-kernel_size+1,stride))
    fig,ax=plt.subplots(figsize=(11,4))
    ax.bar(np.arange(padded_length),np.ones(padded_length),color=['lightgray' if i<padding or i>=padding+length else 'steelblue' for i in range(padded_length)])
    colors=plt.cm.tab10(np.linspace(0,1,max(1,len(starts))))
    for row,(start,color) in enumerate(zip(starts,colors)):
        ax.plot([start-0.4,start+kernel_size-0.6],[1.25+row*0.12]*2,linewidth=5,color=color)
    out_len=len(starts)
    ax.set_xlim(-0.7,padded_length-0.3);ax.set_ylim(0,1.5+len(starts)*0.12)
    ax.set_xlabel('Padded input position');ax.set_ylabel('')
    ax.set_title(f'L={length}, kernel={kernel_size}, stride={stride}, padding={padding} → output length={out_len}')
    plt.show()
widgets.interact(draw_windows,
    length=widgets.IntSlider(value=12,min=5,max=20,step=1,description='Length'),
    kernel_size=widgets.IntSlider(value=3,min=1,max=7,step=2,description='Kernel'),
    stride=widgets.IntSlider(value=1,min=1,max=4,step=1,description='Stride'),
    padding=widgets.IntSlider(value=0,min=0,max=3,step=1,description='Padding'))

### 交互题 A

1. L=12、kernel=3、stride=1、padding=0，输出长度是多少？
2. padding 改为 1 后，输出长度是多少？
3. stride 改为 2 后，窗口数量增加还是减少？
4. kernel 越大，每个输出看到的局部范围怎样变化？

## 4. 输出长度公式

$$L_{out}=\left\lfloor\frac{L_{in}+2p-d(k-1)-1}{s}+1\right\rfloor$$

其中 k=kernel size，s=stride，p=padding，d=dilation。当前先使用 dilation=1。

In [ ]:
def conv_output_length(length,kernel_size,stride=1,padding=0,dilation=1):
    length=torch.as_tensor(length)
    return torch.div(length+2*padding-dilation*(kernel_size-1)-1,stride,rounding_mode='floor')+1

for settings in [(12,3,1,0),(12,3,1,1),(12,3,2,1),(13,5,2,2)]:
    L,k,s,p=settings
    print(settings,'->',int(conv_output_length(L,k,s,p)))

## 5. Conv1d 的输入 shape

我们的 Log-Mel 是 `(B,T,F)`，但 PyTorch Conv1d 期望 `(B,C,L)`：

- B：batch
- C：channels，这里是 40 个 Mel 特征
- L：时间长度

所以需要 `transpose(1,2)`：`(B,T,40) → (B,40,T)`。

In [ ]:
# 准备十条真实语音
features=[]
for digit in range(10):
    audio,sr=sf.read(PARTS/f'{digit}_jackson_0.wav',dtype='float32')
    if audio.ndim>1:audio=audio.mean(axis=1)
    mel=librosa.feature.melspectrogram(y=audio,sr=sr,n_fft=256,win_length=200,hop_length=80,center=False,power=2,n_mels=40)
    features.append(librosa.power_to_db(mel,ref=np.max,top_db=80).T.astype(np.float32))
lengths=np.array([len(v) for v in features],dtype=np.int64)
max_t=int(lengths.max())
batch=np.zeros((10,max_t,40),dtype=np.float32)
mask=np.arange(max_t)[None,:]<lengths[:,None]
valid=np.concatenate(features,axis=0);mean=valid.mean(0);std=valid.std(0)+1e-5
for i,v in enumerate(features):batch[i,:len(v)]=(v-mean)/std
x=torch.from_numpy(batch);x_lengths=torch.from_numpy(lengths);x_mask=torch.from_numpy(mask)
x_conv=x.transpose(1,2)
print('(B,T,F):',x.shape)
print('(B,C,L):',x_conv.shape)

## 6. 一层 Conv1d：看到局部上下文但不缩短时间

使用 kernel=3、stride=1、padding=1。奇数 kernel 的两侧 padding 设为 1，可保持最大序列长度不变。

In [ ]:
conv_context=nn.Conv1d(in_channels=40,out_channels=64,kernel_size=3,stride=1,padding=1)
context=conv_context(x_conv)
print('input:',x_conv.shape)
print('output:',context.shape)
print('weight:',conv_context.weight.shape)
print('bias:',conv_context.bias.shape)
print('参数数量:',sum(p.numel() for p in conv_context.parameters()))

权重 shape `(64,40,3)` 表示 64 个输出通道；每个输出通道同时查看 40 个输入 Mel 通道和连续 3 个时间位置。

如果 Log-Mel frame=25 ms、hop=10 ms，连续 3 帧覆盖的原始时间跨度约为：

$$25+(3-1)\times10=45\text{ ms}$$

## 7. stride=2：时间下采样

stride=2 表示 kernel 每次移动两帧，输出时间长度约减半。计算量下降，但时间分辨率也降低。

In [ ]:
conv_stride2=nn.Conv1d(40,64,kernel_size=3,stride=2,padding=1)
downsampled=conv_stride2(x_conv)
new_lengths=conv_output_length(x_lengths,3,2,1)
print('input max time:',x_conv.shape[-1])
print('output max time:',downsampled.shape[-1])
print('input lengths:',x_lengths.tolist())
print('output lengths:',new_lengths.tolist())

In [ ]:
fig,axes=plt.subplots(2,1,figsize=(12,6),sharex=False)
axes[0].imshow(x[0].T,origin='lower',aspect='auto',cmap='magma');axes[0].set_title(f'Input Log-Mel: T={x_lengths[0]}')
axes[1].imshow(downsampled[0].detach().numpy(),origin='lower',aspect='auto',cmap='coolwarm');axes[1].set_title(f'Conv stride=2: T={new_lengths[0]}')
for ax in axes:ax.set_xlabel('Time position');ax.set_ylabel('Feature/channel')
plt.tight_layout();plt.show()

### 练习 2

1. stride=2 后，10 ms 的输入帧间隔大约变成多少 ms？
2. 时间长度约减半后，计算量通常怎样变化？
3. stride 是否越大越好？为什么？

## 8. 下采样后必须更新 mask

输入 mask shape 是 `(B,T_in)`，输出已经变成 `(B,T_out)`，不能继续使用旧 mask。先用卷积长度公式更新每条 length，再重新生成 mask。

In [ ]:
def lengths_to_mask(lengths,max_length):
    return torch.arange(max_length,device=lengths.device)[None,:]<lengths[:,None]

new_mask=lengths_to_mask(new_lengths,downsampled.shape[-1])
masked_downsampled=downsampled*new_mask.unsqueeze(1)
print('old mask:',x_mask.shape)
print('new mask:',new_mask.shape)
print('output:',masked_downsampled.shape)
print('无效位置为 0:',torch.all(masked_downsampled.transpose(1,2)[~new_mask]==0).item())

In [ ]:
fig,axes=plt.subplots(2,1,figsize=(12,5))
axes[0].imshow(x_mask.numpy(),aspect='auto',cmap='gray_r');axes[0].set_title('Input mask')
axes[1].imshow(new_mask.numpy(),aspect='auto',cmap='gray_r');axes[1].set_title('After stride=2 mask')
for ax in axes:ax.set_xlabel('Time position');ax.set_ylabel('Batch item')
plt.tight_layout();plt.show()

## 9. 两层卷积下采样器

两层 stride=2 的卷积将时间长度大约缩短 4 倍。每层都增加局部上下文。最终转回模型常用的 `(B,T,D)`。

In [ ]:
class ConvSubsampler(nn.Module):
    def __init__(self,input_dim=40,hidden_dim=64):
        super().__init__()
        self.conv1=nn.Conv1d(input_dim,hidden_dim,kernel_size=3,stride=2,padding=1)
        self.conv2=nn.Conv1d(hidden_dim,hidden_dim,kernel_size=3,stride=2,padding=1)
        self.norm=nn.LayerNorm(hidden_dim)

    def forward(self,features,lengths):
        h=features.transpose(1,2)
        h=torch.relu(self.conv1(h))
        lengths=conv_output_length(lengths,3,2,1)
        h=torch.relu(self.conv2(h))
        lengths=conv_output_length(lengths,3,2,1)
        h=self.norm(h.transpose(1,2))
        mask=lengths_to_mask(lengths,h.shape[1])
        h=h*mask.unsqueeze(-1)
        return h,lengths,mask

subsampler=ConvSubsampler()
hidden,out_lengths,out_mask=subsampler(x,x_lengths)
print('input:',x.shape)
print('hidden:',hidden.shape)
print('input lengths:',x_lengths.tolist())
print('output lengths:',out_lengths.tolist())
print('mask:',out_mask.shape)

## 10. 两层卷积的感受野

感受野表示一个输出位置最多依赖多少个原始输入帧。

两层 kernel=3、stride=2：

- 第一层感受野：3 帧，输出步距对应 2 个输入帧。
- 第二层再覆盖 3 个第一层位置。
- 总感受野：$3+(3-1)\times2=7$ 个输入帧。

若输入 hop=10 ms、frame=25 ms，覆盖原始音频跨度约：$25+(7-1)\times10=85$ ms。

In [ ]:
def receptive_field(layers=2,kernel_size=3,stride=2):
    field=1;jump=1
    history=[]
    for layer in range(layers):
        field=field+(kernel_size-1)*jump
        jump*=stride
        history.append((layer+1,field,jump))
    for layer,field,jump in history:print(f'layer {layer}: receptive field={field} input frames, output jump={jump}')
    print(f'对应音频跨度约 {25+(field-1)*10} ms')
widgets.interact(receptive_field,
    layers=widgets.IntSlider(value=2,min=1,max=5,step=1,description='Layers'),
    kernel_size=widgets.IntSlider(value=3,min=1,max=7,step=2,description='Kernel'),
    stride=widgets.IntSlider(value=2,min=1,max=3,step=1,description='Stride'))

### 交互题 B

1. 两层 kernel=3、stride=1 的感受野是多少帧？
2. 两层 kernel=3、stride=2 的时间压缩倍数是多少？
3. 增加层数时感受野怎样变化？
4. 感受野大是否意味着模型一定能有效利用所有位置？

## 11. Padding 的两种含义不要混淆

- Batch padding：把不同长度样本补到同一 T，需要 mask。
- Conv padding：在每条序列两端临时补值，用于控制输出长度和边界计算。

二者都可能是 0，但目的不同。卷积处理完后仍要根据每条真实 length 生成输出 mask。

## 本课测试

1. Conv1d 为什么能利用局部时间上下文？
2. kernel size 表示什么？
3. stride 表示什么？
4. padding=1 对 kernel=3、stride=1 有什么常见作用？
5. PyTorch Conv1d 输入的 shape 顺序是什么？
6. 如何把 `(B,T,F)` 转换成 Conv1d 输入？
7. `Conv1d(40,64,3)` 的权重 shape 是什么？
8. 25 ms frame、10 ms hop、kernel=3 覆盖约多少毫秒？
9. stride=2 后时间帧间隔大约从 10 ms 变成多少？
10. 为什么下采样后必须更新 lengths 和 mask？
11. 两层 stride=2 大约将时间缩短几倍？
12. 两层 kernel=3、stride=2 的感受野是多少输入帧？
13. batch padding 与 convolution padding 有什么区别？
14. Conv1d 输出最后为什么转回 `(B,T,D)`？
15. 局部卷积能否直接看到整句话的任意远位置？

参考答案：1 kernel 同时覆盖相邻帧；2 每次覆盖的位置数；3 每次移动距离；4 保持时间长度；5 `(B,C,L)`；6 transpose(1,2)；7 `(64,40,3)`；8 45 ms；9 20 ms；10 输出时间轴改变；11 4 倍；12 7 帧；13 组成 batch/控制卷积边界；14 便于后续序列模型；15 不能，只看有限感受野。

## 小结

`Log-Mel (B,T,40) → transpose → Conv1d → local context + stride subsampling → transpose → hidden (B,T/4,64) + new lengths + new mask`

卷积擅长局部模式和降低时间长度，但有限层卷积不能直接关联任意远的帧。下一课学习 Self-Attention：让每个时间位置根据内容选择整段序列中的其他位置。

<!-- course-upgrade-v2 -->
## 强化练习：第 9 课专属题库

请先把答案写进新的 Markdown/Code cell，再展开自评标准。

### A. 基础回忆

1. 不看上文，分别定义 `Conv1d`、`stride 下采样`、`感受野`。
2. 哪一个量/状态是本课最容易在模块边界丢失的？它的单位和 shape 是什么？
3. 本课至少写出两个“看起来能运行，但结果其实错误”的例子。

### B. 预测与推理

4. 场景：**连续两层 stride=2 后忘记更新 length**。先预测现象，再说明原因，最后给出一项可以验证猜测的指标。
5. 改变本课最关键参数的 0.5×、1×、2×，分别预测准确率、延迟、内存或数值误差怎样变化。
6. 画一张最小数据流图，在每条边标出 dtype、shape、时间单位或概率/代价方向。

### C. 编程与排错

7. 编程任务：**计算任意卷积栈的输出长度和感受野**。至少加入正常、边界、错误输入三类测试。
8. 故意制造一个 off-by-one、shape、状态未 reset 或数值稳定性错误；记录错误现象和定位过程。
9. 不看本课实现，从空白 cell 重写最核心函数，并用原实现作数值对照。

### D. 迁移与表达

10. 跨课任务：**连接下采样比例与 CTC 最短路径**。
11. 用 90 秒向没有学过 ASR 的人解释本课；禁止只念术语，必须举一个数字或生活例子。
12. 写出一个生产系统中会监控的指标，以及它异常时优先检查的三处位置。

<details><summary>展开自评标准</summary>

- 每题 0～2 分：0=无法回答；1=方向正确但缺少单位、边界或验证；2=解释完整且能用代码/数字验证。
- 24 分满分：达到 19 分再进入下一课；15～18 分次日重做错题；低于 15 分回看本课图和核心代码。
- 第 4 题必须包含“预测—原因—指标”，第 7～9 题必须真正运行测试，第 10 题必须明确上下游 contract。
- 核心答案至少应正确使用：Conv1d、stride 下采样、感受野。

</details>


<!-- course-upgrade-v2 -->
## 间隔复习与离场票

### 离场票（现在完成）

- [ ] 我能不用笔记解释 Conv1d、stride 下采样、感受野。
- [ ] 我能说出本课最常见的错误及其观测现象。
- [ ] 我能从空白重写一个核心函数，并通过至少 3 个测试。
- [ ] 我能说明本课对上一层和下一层接口的影响。

### 复习时间表

- **明天（5 分钟）**：闭卷写出三个核心概念和一个公式/shape。
- **7 天后（15 分钟）**：重做第 4、7、10 题，不运行原答案。
- **30 天后（20 分钟）**：从真实音频或随机张量重新构造一个最小实验。

把错题记录到根目录 `LEARNING_LOG.md`。不要只写“不会”，要写：原判断、证据、正确规则、下次检查动作。
